# Lesson 10 - AI Agents in Production

在本課程中，您將學習使用 Microsoft Agent Framework（Python）的 AI 代理 <strong>生產模式</strong>。我們涵蓋：

- <strong>可觀察性</strong> — 為代理互動新增計時與日誌記錄
- <strong>評估</strong> — 使用評估代理評分回應品質
- <strong>成本管理</strong> — 代幣優化及模型選擇策略

此場景是一個協助用戶規劃行程的 <strong>旅遊代理</strong>，並在其上層實施監控與評估。


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 生產考慮

將 AI 代理從原型轉移到生產階段需要仔細注意三個支柱：

1. <strong>可觀察性</strong> — 你需要了解代理的行為、所需時間以及調用的工具。沒有追蹤和日誌記錄，調試生產問題幾乎不可能。

2. <strong>評估</strong> — 自動化質量檢查確保代理的回應隨時間保持準確、完整且有幫助。評估代理可以根據定義的標準對回應進行評分。

3. <strong>成本管理</strong> — 令牌使用直接影響成本。諸如提示優化、模型選擇和緩存等策略有助於在不犧牲質量的情況下控制開支。


## 建立一個可觀察的代理

我們定義旅遊工具並用計時包裹代理調用，以便我們能監測延遲。在生產環境中，你會整合 OpenTelemetry 或類似的追蹤後端。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評估模式

一個常見的生產模式是使用第二個代理作為<strong>評估者</strong>。評估者根據預先定義的標準，如完整性、準確性和有用性，對主要代理的回應進行評分。

這可實現：
- 在回應到達用戶之前進行自動化質量門檻檢查
- 在提示詞或模型更改時進行回歸檢測
- 持續監控代理的性能表現


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本對於生產 AI 代理至關重要。以下是主要策略：

| Strategy | Description |
|---|---|
| <strong>提示優化</strong> | 保持系統指令簡明扼要。移除冗餘上下文以減少輸入代幣數量。 |
| <strong>模型選擇</strong> | 對於分類或提取等簡單任務，使用較小且便宜的模型（例如 GPT-4o-mini），並將較大模型保留給複雜推理任務。 |
| <strong>緩存</strong> | 緩存工具結果和常用查詢，以避免重複的 API 調用。 |
| <strong>代幣預算</strong> | 設置 `max_tokens` 限制，防止產生意外冗長的回應。 |
| <strong>批次處理</strong> | 盡可能將多個用戶查詢合併為一次 API 調用。 |

在實踐中，分層方法效果良好：將簡單請求路由到快速且經濟的模型，僅將複雜查詢升級到更強大的模型。


## Summary

在本課中，您學會了如何：

1. <strong>新增可觀察性</strong>於代理互動中，通過計時和日誌記錄，為追蹤和監控奠定基礎。
2. <strong>自動評估代理回應</strong>，使用評估代理自動評分完整性、準確性與有用性。
3. <strong>管理成本</strong>，通過提示優化、模型選擇、快取和代幣預算。

這些生產模式有助於確保您的 AI 代理在大規模環境下可靠、可衡量且具成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
